# Task 1.2: Key Assumptions

**Paper:** Improving the Fisher Kernel for Large-Scale Image Classification  
**Authors:** Perronnin, Sánchez, Mensink (ECCV 2010)  
**Student:** Prince Sahoo (230060)

## Assumption 1

**Assumption:** The local descriptors extracted from an image are generated independently from the GMM. In other words, knowing what descriptor was extracted at one location tells you nothing about what descriptor will appear at another location.

**Why the method needs it:** The entire Fisher Vector computation boils down to averaging per-descriptor gradients (Equation 5: $\mathcal{G}_\lambda^X = \frac{1}{T} \sum_t \nabla_\lambda \log u_\lambda(x_t)$). This averaging trick only works if the descriptors are independent — otherwise you'd need to model the joint distribution of all $T$ descriptors together, which would make the computation completely intractable and produce a variable-length representation.

**Violation scenario:** Think about a close-up image of a brick wall or a chessboard — the patches heavily overlap (they're sampled every 16 pixels from 32×32 patches), so adjacent SIFT descriptors share literal pixel data and are highly correlated. The Fisher Vector would treat these redundant descriptors as if each one carries independent information, effectively over-counting repeated patterns.

**Paper reference:** Section 2, directly before Equation 5 — the paper states *"We assume that the $x_t$'s are generated independently by $u_\lambda$."*

## Assumption 2

**Assumption:** Each Gaussian in the GMM has a diagonal covariance matrix — meaning the feature dimensions within each cluster are uncorrelated.

**Why the method needs it:** With diagonal covariances, the Fisher Information Matrix becomes diagonal too, so the normalization in Equations 7–8 simplifies to a per-dimension scaling (just dividing by $\sigma_i$ and $\sqrt{w_i}$). Full covariance would mean the Fisher Vector has $K \cdot D(D+1)/2$ entries instead of $2KD$, and you'd need way more training data to estimate $256 \times 64 \times 65 / 2 \approx 500{,}000$ covariance parameters reliably. The paper applies PCA before the GMM partly to decorrelate features so this assumption holds better.

**Violation scenario:** If you skip the PCA step and feed raw 128-D SIFT descriptors directly, the feature dimensions are quite correlated (e.g., neighboring orientation bins respond to similar edges). Within each Gaussian cluster, these correlations persist, meaning the diagonal assumption would miss inter-dimension relationships that carry discriminative information.

**Paper reference:** Section 2 — *"We assume that the covariance matrices are diagonal and we denote by $\sigma_i^2$ the variance vector."* The closed-form expressions in Equations 7 and 8 explicitly depend on this assumption.

## Assumption 3

**Assumption:** An image's descriptor distribution can be split into two parts: a generic "background" component that follows the trained GMM $u_\lambda$, and an image-specific component $q$. Formally: $p(x) = \omega \cdot q(x) + (1 - \omega) \cdot u_\lambda(x)$ where $\omega$ is the fraction of distinctive content.

**Why the method needs it:** This is the entire theoretical justification for L2 normalization. The derivation in Section 3.1 shows that the background part's gradient vanishes (Equation 12, because the GMM was ML-estimated), so the Fisher Vector reduces to $\omega \times (\text{object-specific signal})$. Since $\omega$ varies with how much background each image has, L2 normalization divides it out. If this two-part decomposition doesn't actually hold for your images, then L2 normalization is dividing out something that isn't just $\omega$ — it could be throwing away useful magnitude information.

**Violation scenario:** Imagine applying this method to satellite imagery or microscopy images where there's no clean "background vs. object" distinction — the entire image is the content of interest. The GMM trained on natural images wouldn't model any part of these images well, so the background contribution wouldn't vanish as Equation 12 assumes. L2 normalization would then normalize away actual signal, not just a scale factor.

**Paper reference:** Section 3.1, Equation 10 defines the decomposition, Equations 11–13 derive the consequence. The paper acknowledges a trade-off: *"This is not to say that the L2 norm of the Fisher vector is not discriminative"* — admitting that the norm they discard may contain class-specific information.

## Assumption 4

**Assumption:** The power normalization exponent $\alpha = 0.5$ works well universally for $K = 256$ Gaussians, regardless of the specific dataset or visual domain.

**Why the method needs it:** Power normalization fixes the sparsity problem in Fisher Vectors (Figure 1 shows the distribution becoming more and more peaked at zero as $K$ grows). But the "right" amount of de-sparsification depends on how sparse the vectors actually are, which in turn depends on $K$, the number of descriptors per image, and the visual complexity. The paper fixes $\alpha = 0.5$ for all experiments to keep things simple. If this one-size-fits-all value is wrong for your specific setting, the normalization could over-compress or under-compress the values.

**Violation scenario:** Consider very small, low-resolution images (like 32×32 thumbnails) that produce only 10–20 SIFT descriptors each. With so few descriptors, most Gaussians actually *do* get non-trivial assignments, so the Fisher Vector isn't sparse at all. Applying $\alpha = 0.5$ would squash values that are already well-distributed, reducing the dynamic range and hurting classification.

**Paper reference:** Section 3.2 — *"In preliminary experiments, we found that $\alpha = 0.5$ was a reasonable value for $K = 256$ and this value is fixed throughout our experiments."* Also Equation 15 and Figure 1(d).